# GPU and evaluation stages on AWS SageMaker (ml.g6.2xlarge, NVIDIA L4 24GB)

Kernel: **conda_pytorch**. Work inside `/home/ec2-user/SageMaker` (only this folder survives a stop).

Setup once: upload `aws_upload.zip` (project code, course data, BM25 negatives) to `/home/ec2-user/SageMaker` with the JupyterLab upload button, then run the cells below in order. Every start: run cells 1 and 2 again (packages are lost on stop), then continue from the next unfinished step. Finished steps are skipped by `run_gpu.sh`.

Only `google-bert/bert-base-uncased` at its verified revision can be loaded (`docs/model_eligibility.md`). No review text is sent to any external service.

**Stop the notebook instance in the AWS console when you are done** (closing the tab does not stop billing).

In [ ]:
# Cell 1: unpack the project (first time only; skipped if already unpacked)
import os, zipfile
BASE = '/home/ec2-user/SageMaker'
PROJECT = f'{BASE}/DATA304_final_project'
if not os.path.isdir(PROJECT):
    zipfile.ZipFile(f'{BASE}/aws_upload.zip').extractall(PROJECT)
%cd $PROJECT
!ls

In [ ]:
# Cell 2: environment (run on every start)
import os, sys
%cd /home/ec2-user/SageMaker/DATA304_final_project
os.environ['HF_HOME'] = '/home/ec2-user/SageMaker/hf'   # keep the model download across stops
os.environ['PYTHON_BIN'] = sys.executable               # use the conda_pytorch interpreter in the shell scripts
%pip install -q "transformers>=4.40"
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers, numpy
print(sys.version.split()[0], torch.__version__, transformers.__version__, numpy.__version__, torch.cuda.is_available())
!{sys.executable} -m unittest discover -s tests

## Status check
After a browser refresh the running cell keeps going on the server, but its output is no longer shown. Run this cell, or open File -> New -> Terminal and run `tail -f ~/SageMaker/DATA304_final_project/outputs/run.log`.

In [ ]:
# Status check: safe to run at any time (also after a browser refresh)
import json, os, subprocess
def done(log):
    return os.path.exists(log) and json.load(open(log)).get('completed', False)
checks = {
    'step 1 zero-shot embeddings': all(os.path.exists(f'outputs/dense/bert-base-uncased/{f}') for f in ('catalog.npy', 'train_reviews.npy', 'test_reviews.npy')),
    'step 2 model A trained': done('outputs/ltr/bert-ltr-dev/train_log.json'),
    'step 2 model A embeddings': os.path.exists('outputs/dense/bert-ltr-dev/train_reviews.npy'),
    'step 3 model B trained': done('outputs/ltr/bert-ltr-full/train_log.json'),
    'step 3 model B embeddings': os.path.exists('outputs/dense/bert-ltr-full/test_reviews.npy'),
    'step 4 zero-shot evaluation': os.path.exists('outputs/dense_eval/bert-base-uncased/metrics.json'),
    'step 4 LTR evaluation': os.path.exists('outputs/dense_eval/bert-ltr/metrics.json'),
    'step 4b candidates': os.path.exists('outputs/rerank/candidates_test.json'),
    'step 4b cross-encoder A trained': done('outputs/rerank/ce-dev/train_log.json'),
    'step 4b validation scored': os.path.exists('outputs/rerank/ce-dev/scores_validation.npy'),
    'step 4b cross-encoder B trained': done('outputs/rerank/ce-full/train_log.json'),
    'step 4b test scored': os.path.exists('outputs/rerank/ce-full/scores_test.npy'),
    'step 4b evaluation': os.path.exists('outputs/rerank/submission/metrics.json'),
    'step 4c pseudo-labels (dev)': os.path.exists('outputs/pseudo/pseudo_dev.json'),
    'step 4c self-trained A': done('outputs/pseudo/ce-dev/train_log.json'),
    'step 4c validation scored': os.path.exists('outputs/pseudo/ce-dev/scores_validation.npy'),
    'step 4c pseudo-labels (test)': os.path.exists('outputs/pseudo/pseudo_full.json'),
    'step 4c self-trained B': done('outputs/pseudo/ce-full/train_log.json'),
    'step 4c test scored': os.path.exists('outputs/pseudo/ce-full/scores_test.npy'),
    'step 4c evaluation': os.path.exists('outputs/pseudo/submission/metrics.json'),
    'step 4d model-B train embeddings': os.path.exists('outputs/dense/bert-ltr-full/train_reviews.npy'),
    'step 4d category model A': done('outputs/category/dev/train_log.json'),
    'step 4d category model B': done('outputs/category/full/train_log.json'),
    'step 4d evaluation': os.path.exists('outputs/category/submission/metrics.json'),
    'step 4e ANN (IVF) evaluation': os.path.exists('outputs/ann/metrics.json'),
    'step 4f hybrid negatives': os.path.exists('outputs/hardneg/negatives_full.json'),
    'step 4f cross-encoder A trained': done('outputs/hardneg/ce-dev/train_log.json'),
    'step 4f validation scored': os.path.exists('outputs/hardneg/ce-dev/scores_validation.npy'),
    'step 4f cross-encoder B trained': done('outputs/hardneg/ce-full/train_log.json'),
    'step 4f test scored': os.path.exists('outputs/hardneg/ce-full/scores_test.npy'),
    'step 4f evaluation': os.path.exists('outputs/hardneg/submission/metrics.json'),
    'step 4g proxy candidates': os.path.exists('outputs/memory/candidates_mem_test.json'),
    'step 4g scores (of 12)': len(__import__('glob').glob('outputs/memory/scores/*.npy')),
    'step 4g evaluation': os.path.exists('outputs/memory/submission/metrics.json'),
    'step 4h model A': os.path.exists('outputs/augce/ce-dev/train_log.json') and json.load(open('outputs/augce/ce-dev/train_log.json')).get('completed', False),
    'step 4h model B': os.path.exists('outputs/augce/ce-full/train_log.json') and json.load(open('outputs/augce/ce-full/train_log.json')).get('completed', False),
    'step 4h scores (of 3)': len(__import__('glob').glob('outputs/memory/scores/aug_exp_*.npy')),
    'step 4h evaluation': os.path.exists('outputs/augce/submission/metrics.json'),
}
for name, ok in checks.items(): print(('DONE   ' if ok else 'not yet') if isinstance(ok, bool) else f'{ok:>7}', name)
print(subprocess.run("ps aux | grep -E 'encode_dense|train_dense_ltr|run_dense|mine_negatives|make_candidates|cross_encoder|run_rerank|pseudo|category|run_ann|hybrid_negatives|memory' | grep -v grep", shell=True, capture_output=True, text=True).stdout or 'no pipeline process running')
!tail -n 5 outputs/run.log

## Step 1: dense retrieval, zero-shot BERT embeddings (about 10-15 minutes)

In [ ]:
!bash run_gpu.sh zeroshot 2>&1 | tee -a outputs/run.log

## Step 2: learning to rank, model A (12,000 queries; epoch chosen on the tuning slice) and its embeddings

In [ ]:
!bash run_gpu.sh ltr-dev 2>&1 | tee -a outputs/run.log
import json
log = json.load(open('outputs/ltr/bert-ltr-dev/train_log.json'))
for e in log['epochs']: print(e)
print('selected epoch:', log['selected_epoch'])

## Step 3: learning to rank, model B (all 20,000 queries, stops at model A's epoch) and test embeddings

In [ ]:
!bash run_gpu.sh ltr-full 2>&1 | tee -a outputs/run.log

## Step 4: evaluation and Kaggle files (CPU, NumPy; about 15-25 minutes)

In [ ]:
!bash run_dense_eval.sh 2024320020 8 all 2>&1 | tee -a outputs/run.log

## Step 4b: multi-stage ranking, BERT cross-encoder reranker (about 4-5 hours; each part can be re-run)
Candidates (CPU, ~15 min) -> model A training with epoch selection + validation scores -> model B training + test scores -> evaluation and Kaggle file.

In [ ]:
!bash run_rerank.sh 2024320020 8 candidates 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_rerank.sh 2024320020 8 dev 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_rerank.sh 2024320020 8 full 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_rerank.sh 2024320020 8 eval 2>&1 | tee -a outputs/run.log

## Step 4c: self-training with pseudo-labels (about 4-5 hours; each part can be re-run)
Confident predictions of the step-4b ranker on unlabeled queries become extra cross-encoder training pairs. Pseudo-labels for tuning slice + validation (CPU, ~1 min) -> self-trained model A + validation scores -> pseudo-labels for test (CPU) -> self-trained model B + test scores -> evaluation and Kaggle file.

In [ ]:
!bash run_pseudo.sh 2024320020 labels-dev 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_pseudo.sh 2024320020 dev 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_pseudo.sh 2024320020 labels-full 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_pseudo.sh 2024320020 full 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_pseudo.sh 2024320020 eval 2>&1 | tee -a outputs/run.log

## Step 4d: category labels as auxiliary information (label GCN, multi-task; about 10-20 minutes)
Model-B training-review embeddings (GPU, ~2 min) -> category model A (epoch on the tuning slice) -> category model B -> (w, beta) on the tuning slice, validation once, Kaggle file.

In [ ]:
!bash run_category.sh 2024320020 encode 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_category.sh 2024320020 dev 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_category.sh 2024320020 full 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_category.sh 2024320020 eval 2>&1 | tee -a outputs/run.log

## Step 4e: ANN with an IVF index, speed vs. accuracy (CPU, NumPy; about 5-10 minutes)

In [ ]:
!{sys.executable} scripts/run_ann.py 2>&1 | tee -a outputs/run.log

## Step 4f: cross-encoder with hybrid hard negatives (about 3 hours; each part can be re-run)
Same reranker as step 4b, but training negatives come from the hybrid BM25 v2 + LTR BERT first stage (the reranker's own candidates). Hybrid negatives (CPU, ~5 min) -> model A + validation scores -> model B + test scores -> evaluation and Kaggle file. Needs step 4b and step 4d `encode`.

In [ ]:
!bash run_hardneg.sh 2024320020 8 mine 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_hardneg.sh 2024320020 8 dev 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_hardneg.sh 2024320020 8 full 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_hardneg.sh 2024320020 8 eval 2>&1 | tee -a outputs/run.log

## Step 4g: test-like proxy validation with labelled-review memory (about 1.5-2 hours; no training)
Proxy splits + memory first stage (CPU, ~15-25 min) -> cross-encoder scoring of every candidate set (GPU, ~1 hour) -> selection on the proxy tuning slice, proxy validation once, Kaggle files. Needs steps 4b, 4d `encode` and 4f.

In [ ]:
!bash run_memory.sh 2024320020 8 candidates 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_memory.sh 2024320020 8 score 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_memory.sh 2024320020 8 eval 2>&1 | tee -a outputs/run.log

## Step 4h: augmented cross-encoder (about 4-4.5 hours; each part can be re-run)
Shared 320-token review budget, review shuffling/subsets during training, 4 hybrid + 3 BM25 negatives per group. Needs steps 4b, 4f and 4g (`candidates`). Run the four cells in order; if the browser disconnects, re-run the same cell (finished parts are skipped).

In [ ]:
!bash run_augce.sh 2024320020 dev 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_augce.sh 2024320020 full 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_augce.sh 2024320020 score 2>&1 | tee -a outputs/run.log

In [ ]:
!bash run_augce.sh 2024320020 eval 2>&1 | tee -a outputs/run.log

## Step 5: pack the small results for download (metrics, logs, submission CSVs; no embeddings)

In [ ]:
import glob, zipfile
files = (glob.glob('outputs/dense_eval/**/*', recursive=True)
         + glob.glob('outputs/ltr/bert-ltr-*/train_log.json')
         + glob.glob('outputs/ltr/bert-ltr-*/checkpoint/provenance.json')
         + glob.glob('outputs/dense/*/manifest.json')
         + glob.glob('outputs/rerank/submission/*')
         + glob.glob('outputs/rerank/ce-*/train_log.json')
         + glob.glob('outputs/rerank/ce-*/checkpoint/provenance.json')
         + glob.glob('outputs/pseudo/*_summary.json')
         + glob.glob('outputs/pseudo/submission/*')
         + glob.glob('outputs/pseudo/ce-*/train_log.json')
         + glob.glob('outputs/pseudo/ce-*/checkpoint/provenance.json')
         + glob.glob('outputs/category/submission/*')
         + glob.glob('outputs/category/*/train_log.json')
         + glob.glob('outputs/ann/metrics.json')
         + glob.glob('outputs/hardneg/*_summary.json')
         + glob.glob('outputs/hardneg/submission/*')
         + glob.glob('outputs/hardneg/ce-*/train_log.json')
         + glob.glob('outputs/hardneg/ce-*/checkpoint/provenance.json')
         + glob.glob('outputs/memory/memory_selection.json')
         + glob.glob('outputs/memory/submission/**/*', recursive=True)
         + glob.glob('outputs/memory/scores/*.npy')
         + glob.glob('outputs/memory/candidates_*.json')
         + glob.glob('outputs/augce/submission/*')
         + glob.glob('outputs/augce/ce-*/train_log.json')
         + glob.glob('outputs/augce/ce-*/checkpoint/provenance.json'))
with zipfile.ZipFile('results_download.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if os.path.isfile(f):
            z.write(f)
print(len(files), 'files ->', os.path.abspath('results_download.zip'))

Right-click `DATA304_final_project/results_download.zip` in the file browser -> Download, and unzip it into the local project folder. Then **stop the instance** in the console.

## Step 6: final file, model pack, and a clean quick-path reproduction check (about 40-50 minutes)
6a writes `outputs/final/2024320020_final.csv` (exp + ens, Kaggle 0.77188) and packs the three model-B checkpoints with a SHA-256 manifest.
6b copies only the code, the committed records and the data into a fresh folder, and runs `bash run_all.sh quick` there with the packed models, as a grader would.
6c compares the reproduced file with the submitted one.

In [ ]:
!python scripts/make_final.py --student-id 2024320020 2>&1 | tee -a outputs/run.log
!python scripts/validate_submission.py outputs/final/2024320020_final.csv
!python scripts/pack_models.py --student-id 2024320020 2>&1 | tee -a outputs/run.log

In [ ]:
import os, shutil
R = '/home/ec2-user/SageMaker/repro_check'
shutil.rmtree(R, ignore_errors=True)
os.makedirs(R)
for d in ['scripts', 'src', 'tests', 'artifacts']:
    shutil.copytree(d, f'{R}/{d}', ignore=shutil.ignore_patterns('__pycache__'))
for f in os.listdir('.'):
    if f.endswith('.sh') or f.startswith('requirements'):
        shutil.copy(f, R)
os.makedirs(f'{R}/project_release')
for f in os.listdir('project_release'):
    shutil.copy(f'project_release/{f}', f'{R}/project_release/{f}')
shutil.copy('outputs/model_pack/model_manifest.json', f'{R}/artifacts/model_manifest.json')
shutil.copy('outputs/model_pack/models_2024320020.zip', R)
print(sorted(os.listdir(R)))
!cd {R} && bash run_all.sh quick 2024320020 8 2>&1 | tee quick_run.log

In [ ]:
!python scripts/compare_submissions.py /home/ec2-user/SageMaker/repro_check/outputs/final/2024320020_final.csv outputs/final/2024320020_final.csv
!python scripts/compare_submissions.py /home/ec2-user/SageMaker/repro_check/outputs/final/2024320020_final.csv outputs/memory/submission/exp_ens/2024320020_final.csv
import shutil
for f in ['outputs/final/metrics.json', 'outputs/final/2024320020_final.csv', 'outputs/model_pack/model_manifest.json']:
    print(f, os.path.isfile(f))
shutil.copy('/home/ec2-user/SageMaker/repro_check/quick_run.log', 'outputs/final/quick_run.log')
shutil.copy('/home/ec2-user/SageMaker/repro_check/outputs/final/metrics.json', 'outputs/final/quick_metrics.json')
import zipfile
with zipfile.ZipFile('final_download.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['outputs/final/metrics.json', 'outputs/final/2024320020_final.csv', 'outputs/final/quick_run.log',
              'outputs/final/quick_metrics.json', 'outputs/model_pack/model_manifest.json']:
        z.write(f)
print('final_download.zip written')

Download `final_download.zip` (small) and `outputs/model_pack/models_2024320020.zip` (about 1.3 GB, the three checkpoints). Upload the model zip to Google Drive, share it as "anyone with the link can view", and send the link. Then **stop the instance**.